## Notebook 概览: `scripts/pytorch2onnx.py`

`scripts/pytorch2onnx.py` 是一个实用工具脚本，其核心功能是将 Real-ESRGAN 项目中训练好的 PyTorch 模型（通常是以 `.pth` 文件格式存储的权重）转换为 ONNX (Open Neural Network Exchange) 格式。

**核心职责与目的:**

1.  **模型格式转换**: ONNX 是一种为机器学习模型设计的开放标准格式。通过将 PyTorch 模型转换为 ONNX 格式，可以实现：
    *   **互操作性 (Interoperability)**: 允许模型在不同的深度学习框架（如 TensorFlow, MXNet, Caffe2 等）之间进行迁移和使用。
    *   **部署优化 (Deployment Optimization)**: 许多推理引擎和硬件加速器（如 NVIDIA TensorRT, Intel OpenVINO, ONNX Runtime）对 ONNX 格式有良好支持，可以对模型进行进一步优化以在特定硬件上实现高效部署。
    *   **硬件兼容性**: 方便模型在多种硬件平台上运行，包括服务器、边缘设备和移动设备。

2.  **脚本工作流程**: 
    *   **参数解析**: 使用 `argparse` 解析命令行参数，用户需要指定输入的 PyTorch 模型权重文件路径 (`.pth`)、输出的 ONNX 模型文件路径 (`.onnx`)、模型名称（用于确定网络结构）以及其他转换参数（如ONNX算子集版本）。
    *   **PyTorch 模型加载**: 根据指定的模型名称和缩放比例，实例化对应的 Real-ESRGAN 网络架构（如 `RRDBNet` 或 `SRVGGNetCompact`）。然后，加载预训练的 `.pth` 权重文件到这个 PyTorch 模型实例中，并将其设置为评估模式 (`model.eval()`)。
    *   **创建虚拟输入 (Dummy Input)**: ONNX 的导出过程需要一个符合模型输入尺寸和类型的示例张量（虚拟输入）。脚本会创建一个这样的小尺寸随机张量。
    *   **执行导出 (`torch.onnx.export()`)**: 调用 PyTorch 核心的 `torch.onnx.export()` 函数。这个函数会：
        *   “追踪”(trace) 虚拟输入通过 PyTorch 模型的执行路径。
        *   将模型中的操作转换为等效的 ONNX 算子。
        *   将转换后的计算图和模型权重保存为一个 `.onnx` 文件。
    *   **配置导出参数**: 在调用 `torch.onnx.export()` 时，可以指定重要的参数，如 `opset_version`（ONNX算子集版本，影响兼容性）和 `dynamic_axes`（指定输入的哪些维度是动态的，例如图像的高度和宽度，这对于超分辨率模型非常重要，因为它通常需要处理不同尺寸的输入图像）。

**主要依赖:**
*   `torch`: PyTorch 深度学习框架，用于加载原始模型并执行导出操作。
*   `argparse`: Python 标准库，用于解析命令行参数。
*   `os` (及其子模块 `os.path`): 用于文件系统路径操作。
*   Real-ESRGAN 的模型架构定义: 
    *   `basicsr.archs.rrdbnet_arch.RRDBNet`: 用于加载基于 RRDBNet 的 Real-ESRGAN 模型。
    *   `realesrgan.archs.srvgg_arch.SRVGGNetCompact`: 用于加载基于 SRVGGNetCompact 的 Real-ESRGAN 模型变体。

通过这个脚本，开发者可以将训练好的 Real-ESRGAN PyTorch 模型转换为更具可移植性和部署灵活性的 ONNX 格式，便于在多种生产环境和硬件上应用。

In [ ]:
import argparse
import os
import torch

# It's common to place these imports here if they are directly used for model instantiation
from basicsr.archs.rrdbnet_arch import RRDBNet 
from realesrgan.archs.srvgg_arch import SRVGGNetCompact 

# If the project structure requires adding parent directories to sys.path for modules to be found:
# import sys
# from os import path as osp
# sys.path.append(osp.dirname(osp.dirname(osp.abspath(__file__))))


**代码解释：导入模块**

*   `import argparse`:
    *   导入 Python 标准库中的 `argparse` 模块。该模块用于创建命令行界面，使得脚本能够方便地从用户那里接收参数，例如输入的PyTorch模型路径、输出的ONNX模型路径以及模型名称等。

*   `import os`:
    *   导入 Python 内置的 `os` 模块，它提供了与操作系统进行交互的功能。在这个脚本中，虽然直接使用可能不多（更常见的是其子模块 `os.path`），但通常为了路径操作或文件系统检查而被包含进来。

*   `import torch`:
    *   导入 PyTorch 深度学习框架。PyTorch 是此脚本的核心依赖，因为：
        1.  需要用 PyTorch 来加载预训练的 Real-ESRGAN 模型权重（`.pth` 文件）。
        2.  需要实例化 PyTorch 模型对象。
        3.  核心的 `torch.onnx.export()` 函数是 PyTorch 提供的用于将模型导出到 ONNX 格式的工具。

*   `from basicsr.archs.rrdbnet_arch import RRDBNet`:
    *   从 `basicsr` 库（Real-ESRGAN 依赖的基础库）的 `archs.rrdbnet_arch` 模块中导入 `RRDBNet` 类。
    *   RRDBNet (Residual-in-Residual Dense Block Network) 是 Real-ESRGAN 主要使用的生成器网络架构之一（例如 `RealESRGAN_x4plus` 和 `RealESRNet_x4plus` 模型）。脚本需要这个类定义来首先创建一个 PyTorch 模型实例，然后才能将加载的权重填充进去并导出。

*   `from realesrgan.archs.srvgg_arch import SRVGGNetCompact`:
    *   从 `realesrgan` 包自身的 `archs.srvgg_arch` 模块中导入 `SRVGGNetCompact` 类。
    *   SRVGGNetCompact 是一种基于VGG的紧凑型网络架构，用于 Real-ESRGAN 的一些特定模型变体（例如 `realesr-animevideov3`, `realesr-general-x4v3`）。与 `RRDBNet` 类似，脚本需要这个类定义来实例化相应的 PyTorch 模型。

注释掉的 `sys.path.append` 部分展示了一种常见做法：如果脚本依赖的自定义模块（如 `basicsr` 或 `realesrgan` 包自身）没有作为已安装的库存在于 Python 的标准搜索路径中，开发者有时会动态地将项目根目录或相关目录添加到 `sys.path`，以确保 Python 解释器能够找到这些模块。在这个教学示例中，我们假设这些包要么已经安装，要么脚本是从能够正确解析这些导入的项目环境中运行的。

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        '--model_path', 
        type=str, 
        required=True, 
        help='Path to the PyTorch model checkpoint (.pth file)')
    parser.add_argument(
        '--output_path', 
        type=str, 
        required=True, 
        help='Path to save the exported ONNX model (.onnx file)')
    parser.add_argument(
        '--model_name',
        type=str,
        default='RealESRGAN_x4plus', # Default model if not inferable or specified
        help='Model architecture name (e.g., RealESRGAN_x4plus, realesr-animevideov3). '
             'This helps in selecting the correct PyTorch model class.')
    parser.add_argument(
        '--scale', 
        type=int, 
        default=4, 
        help='The native scale of the model (e.g., 2 for x2, 4 for x4)')
    parser.add_argument(
        '--opset_version', 
        type=int, 
        default=11, 
        help='ONNX opset version to use for export.')
    # Potentially add args for dynamic axes if needed for variable input size in ONNX
    # parser.add_argument('--dynamic_axes', action='store_true', help='Enable dynamic axes for input/output.')
    args = parser.parse_args()


In [ ]:
    # Instantiate the model based on args.model_name and args.scale
    model_name_lower = args.model_name.lower()
    if 'realesrgan_x4plus' in model_name_lower or 'realesrnet_x4plus' in model_name_lower:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4) # Scale is fixed for these specific named models
        # Ensure args.scale matches if this specific model is chosen, or warn.
        if args.scale != 4: print(f"Warning: Model {args.model_name} is natively x4, but --scale is {args.scale}")
    elif 'realesrgan_x2plus' in model_name_lower:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
        if args.scale != 2: print(f"Warning: Model {args.model_name} is natively x2, but --scale is {args.scale}")
    elif 'realesrgan_x4plus_anime_6b' in model_name_lower: # Match anime model name
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        if args.scale != 4: print(f"Warning: Model {args.model_name} is natively x4, but --scale is {args.scale}")
    elif 'realesr-animevideov3' in model_name_lower:
        model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4, act_type='prelu')
        if args.scale != 4: print(f"Warning: Model {args.model_name} is natively x4, but --scale is {args.scale}")
    elif 'realesr-general-x4v3' in model_name_lower:
        model = SRVGGNetCompact(num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=32, upscale=4, act_type='prelu')
        if args.scale != 4: print(f"Warning: Model {args.model_name} is natively x4, but --scale is {args.scale}")
    else:
        # Fallback or raise error if model_name is not recognized and specific params are needed
        # For a generic case, one might need to pass num_block, num_feat etc. as args
        print(f"Warning: Model name '{args.model_name}' not recognized for specific architecture. Defaulting to RRDBNet with scale {args.scale}. Ensure this is correct.")
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=args.scale)

    loadnet = torch.load(args.model_path, map_location='cpu')
    if 'params_ema' in loadnet:
        keyname = 'params_ema'
    else:
        keyname = 'params'
    model.load_state_dict(loadnet[keyname], strict=True)
    model = model.cpu() # Ensure model is on CPU for ONNX export


In [ ]:
    # Create a dummy input tensor
    # Input shape (batch_size, channels, height, width)
    # Height and width can be arbitrary for testing, but should reflect typical usage or be small for speed.
    # For SR models, dynamic axes are often preferred for H, W.
    # Using a small dummy input for the export process itself.
    dummy_input = torch.randn(1, 3, 64, 64, device='cpu') 

    # Define input and output names for the ONNX model (optional but good practice)
    input_names = ["input"]
    output_names = ["output"]

    # Export the model
    print(f"Exporting model to {args.output_path} with opset version {args.opset_version}...")
    torch.onnx.export(
        model,
        dummy_input,
        args.output_path,
        verbose=False, # Set to True for detailed export information
        input_names=input_names,
        output_names=output_names,
        opset_version=args.opset_version,
        # Example for dynamic axes if an argument args.dynamic_axes was added:
        # dynamic_axes={'input': {0: 'batch_size', 2: 'height', 3: 'width'},
        #               'output': {0: 'batch_size', 2: 'height', 3: 'width'}} if args.dynamic_axes else None
        # For Real-ESRGAN, typically only H and W are dynamic for input and output.
        # Batch size might be dynamic too if the deployment supports batching.
        dynamic_axes={'input': {2: 'height', 3: 'width'}, 
                      'output': {2: 'height', 3: 'width'}}
    )
    print(f"Model successfully exported to {args.output_path}")

    # ... (rest of main: model loading, ONNX export)

if __name__ == '__main__':
    # main() # We will add the rest of main's logic before calling it
    pass

**代码解释：`main()` 函数定义与参数解析**

`def main():` 定义了脚本的主要执行函数。

**命令行参数解析 (`argparse`)**: 
在 `main()` 函数内部，首先使用 `argparse` 模块来设置和解析命令行参数，这些参数允许用户在运行脚本时指定关键信息。

*   `parser = argparse.ArgumentParser()`: 创建一个 `ArgumentParser` 对象，用于注册参数定义。

*   `parser.add_argument('--model_path', type=str, required=True, help=...)`:
    *   定义一个必需的 (`required=True`) 命令行参数 `--model_path`。
    *   用户需要提供一个字符串 (`type=str`)，该字符串是输入的 PyTorch 模型检查点文件（`.pth` 文件）的路径。
    *   `help` 提供了参数说明。

*   `parser.add_argument('--output_path', type=str, required=True, help=...)`:
    *   定义一个必需的参数 `--output_path`，指定导出的 ONNX 模型文件（`.onnx` 文件）的保存路径。

*   `parser.add_argument('--model_name', type=str, default='RealESRGAN_x4plus', help=...)`:
    *   定义参数 `--model_name`，用于指定要转换的模型的架构名称。
    *   默认值为 `'RealESRGAN_x4plus'`。这个参数非常重要，因为它决定了脚本内部应该实例化哪个 PyTorch 模型类（例如 `RRDBNet` 或 `SRVGGNetCompact`）来加载权重并进行后续的导出操作。

*   `parser.add_argument('--scale', type=int, default=4, help=...)`:
    *   定义参数 `--scale`，用于指定模型原生的放大倍数（例如，对于x4模型，此值为4；对于x2模型，为2）。
    *   这个信息对于正确实例化某些模型架构（这些架构的构造函数可能需要 `scale` 参数）是必需的。

*   `parser.add_argument('--opset_version', type=int, default=11, help=...)`:
    *   定义参数 `--opset_version`，用于指定导出 ONNX 模型时使用的算子集 (operator set) 版本。
    *   ONNX 算子集定义了模型中可以使用的操作（算子）的集合和版本。不同的版本支持不同的算子或算子行为。选择合适的 `opset_version` 对于确保模型能被目标推理引擎正确解析和执行非常重要。默认值通常是一个广泛兼容的版本（如此处的11）。

*   `# parser.add_argument('--dynamic_axes', action='store_true', help=...)`:
    *   这是一个被注释掉的参数示例，用于启用动态轴 (dynamic axes) 功能。
    *   对于图像超分辨率模型，输入图像的高度和宽度通常是可变的。通过设置动态轴，可以使导出的 ONNX 模型能够接受不同尺寸的输入，而不是固定为导出时使用的虚拟输入的尺寸。这对于模型的实际应用非常关键。如果未启用，ONNX模型将只接受与导出时虚拟输入相同尺寸的输入。

*   `args = parser.parse_args()`: 
    *   调用 `parse_args()` 方法来处理命令行中用户实际输入的参数。解析后的参数值将存储在 `args` 对象中，后续可以通过 `args.model_path`, `args.output_path` 等方式访问。

脚本的后续部分（模型加载、ONNX导出）将使用 `args` 中的这些值来执行转换。

**代码解释：模型加载与准备**

在解析命令行参数后，脚本需要根据这些参数（主要是 `args.model_name` 和 `args.scale`）实例化正确的 PyTorch 模型架构，并从指定的 `args.model_path` 加载预训练的权重。

*   **模型实例化**: 
    *   `model_name_lower = args.model_name.lower()`: 将用户提供的模型名称转换为小写，以便进行不区分大小写的比较。
    *   通过一系列 `if/elif` 语句，根据 `model_name_lower` 中包含的特定子字符串来判断应该使用哪个模型类 (`RRDBNet` 或 `SRVGGNetCompact`) 以及相应的配置参数（如 `num_block`，`num_conv`，`scale`）。
        *   例如，如果模型名包含 `'realesrgan_x4plus'`，则实例化一个 `RRDBNet`，其 `scale` 参数被硬编码为4（因为这是该特定模型的原生放大倍数）。
        *   同时，会检查用户通过 `--scale` 参数传入的值是否与模型的原生放大倍数一致，如果不一致，则打印警告。这是因为 ONNX 导出通常针对模型的原生结构进行。
    *   如果 `args.model_name` 未匹配任何已知模式，脚本会打印一条警告，并默认尝试使用 `RRDBNet` 和用户通过 `--scale` 指定的放大倍数来实例化模型。这种回退机制假设用户可能在转换一个自定义的、但与RRDBNet兼容的架构。

*   **加载模型权重**: 
    *   `loadnet = torch.load(args.model_path, map_location='cpu')`: 使用 `torch.load` 从 `args.model_path` 指定的路径加载 `.pth` 文件。`map_location='cpu'` 参数确保权重首先加载到CPU内存，这有助于避免在没有GPU或GPU显存不足的环境中加载失败，同时也方便后续的ONNX导出（通常在CPU上进行）。`loadnet` 此时是一个包含模型参数和其他可能的元数据（如优化器状态）的字典。
    *   **选择权重类型 (EMA vs. standard)**: 
        *   `if 'params_ema' in loadnet: keyname = 'params_ema' else: keyname = 'params'`: 检查加载的 `loadnet` 字典中是否存在键为 `'params_ema'` 的项。EMA (Exponential Moving Average) 权重通常被认为比标准训练权重 (`'params'`) 具有更好的泛化性能和更平滑的输出。因此，如果EMA权重可用，则优先使用它们；否则，使用标准的 `'params'` 权重。
    *   `model.load_state_dict(loadnet[keyname], strict=True)`: 将选定的权重（`loadnet[keyname]`）加载到先前实例化的 PyTorch 模型 `model` 中。`strict=True` 参数表示加载的权重字典中的键必须与模型架构中定义的参数名称完全匹配，不允许有缺失或多余的键。

*   **模型模式与设备设置**: 
    *   `model.eval()`: 将模型设置为评估（推理）模式。这是非常重要的一步，因为它会关闭像 Dropout 这样的训练特定层，并确保 Batch Normalization 等层使用其在训练期间学习到的固定统计数据，而不是当前批次的统计数据。
    *   `model = model.cpu()`: 确保模型位于CPU上。ONNX导出操作通常在CPU上执行，即使模型最终可能在GPU上运行。将模型移到CPU可以避免不必要的GPU内存占用和潜在的设备兼容性问题。

完成这些步骤后，`model` 对象就包含了预训练的权重，并且处于正确的评估模式，准备好被导出到ONNX格式。

**代码解释：ONNX 导出过程**

在模型加载并准备就绪后，此部分代码执行核心的 PyTorch 到 ONNX 的转换操作。

*   **创建虚拟输入 (Dummy Input)**:
    *   `dummy_input = torch.randn(1, 3, 64, 64, device='cpu')`: 
        *   创建一个符合模型输入期望的随机张量。这个张量的具体数值不重要，但其形状 (shape) 和数据类型 (dtype) 必须与模型实际期望的输入一致。
        *   形状 `(1, 3, 64, 64)` 代表：
            *   `1`: 批次大小 (batch_size) 为1。
            *   `3`: 输入图像的通道数 (channels)，例如RGB图像为3。
            *   `64, 64`: 图像的高度 (height) 和宽度 (width)。这里使用64x64是一个示例尺寸，实际应用中，如果模型支持动态尺寸（通过 `dynamic_axes` 设置），这个具体尺寸主要用于追踪计算图。如果模型不支持动态尺寸，则导出的ONNX模型将只能接受这个固定尺寸的输入。
        *   `device='cpu'`: 明确指定这个虚拟输入张量在CPU上创建，与模型所在的设备一致。
    *   **作用**: `torch.onnx.export()` 函数需要一个这样的虚拟输入来执行一次“追踪导出”(trace-based export)。它会模拟一次前向传播，记录下所有执行的操作，并将这些操作转换为ONNX算子。

*   **定义输入输出名称 (Input/Output Names)**:
    *   `input_names = ["input"]` 和 `output_names = ["output"]`: 
        *   为导出的ONNX模型中的输入和输出节点指定名称。这是一个良好的实践，可以使得后续使用ONNX模型（例如在ONNX Runtime中加载和运行）时，能够通过这些易于理解的名称来引用输入和输出张量，而不是依赖自动生成的、可能不直观的节点名称。

*   **执行ONNX导出 (`torch.onnx.export()`)**:
    *   `print(f"Exporting model to {args.output_path}...")`: 打印导出开始的信息。
    *   `torch.onnx.export(...)`: 这是PyTorch提供的核心函数，用于将PyTorch模型转换为ONNX格式。
        *   `model`: 准备好的PyTorch模型实例（已加载权重，并处于 `eval()` 模式）。
        *   `dummy_input`: 上一步创建的虚拟输入张量。
        *   `args.output_path`: 用户指定的输出ONNX文件的路径。
        *   `verbose=False`: 控制导出过程是否打印详细的日志信息。设置为 `True` 可以帮助调试转换过程中的问题。在此示例中设为 `False` 以保持输出简洁。
        *   `input_names=input_names`: 设置ONNX图中输入节点的名称。
        *   `output_names=output_names`: 设置ONNX图中输出节点的名称。
        *   `opset_version=args.opset_version`: 指定要使用的ONNX算子集版本。这影响了哪些PyTorch操作可以被正确转换，以及与不同ONNX运行时环境的兼容性。
        *   `dynamic_axes={'input': {2: 'height', 3: 'width'}, 'output': {2: 'height', 3: 'width'}}`: 
            *   **非常关键的参数**，特别是对于图像超分辨率这类需要处理不同尺寸输入的模型。
            *   它告诉ONNX导出器，输入张量（名为 `'input'`）的第2维（高度）和第3维（宽度）是动态的，可以接受不同大小的值。同样，输出张量（名为 `'output'`）的相应维度也是动态的。
            *   这样配置后，导出的ONNX模型将更加灵活，能够处理任意尺寸的输入图像（在模型架构允许的范围内），而不是仅仅固定为导出时 `dummy_input` 的尺寸。
            *   注释中提到了也可以将批次大小（第0维）设为动态，如果部署环境支持批处理不同大小的批次。

*   `print(f"Model successfully exported to {args.output_path}")`: 打印导出成功的确认信息。

通过这些步骤，脚本能够将训练好的、特定于PyTorch的Real-ESRGAN模型转换为一个标准的、框架无关的ONNX模型文件，为后续的部署和优化提供了便利。